# Kumo Engine with MLOps: train, track, register and score

This notebook starts from the familiar Kumo Engine notebook and turns on MLOps with a self-managed MLflow server.
You will train, then find and use everything that was logged.
It runs on the Brev instance `kumo-engine-experimental`, from the `prototype/result-summary` branch.

- Steps 1 to 4: where you start, install, the MLflow server, turn MLOps on.
- Step 5: train eight small seeded examples, one per task type, into one MLflow experiment.
- Step 6: find the runs, trials, metrics, lineage and models in MLflow, from Python.
- Step 7: promote a model, load it back and score with it.
- Step 8: open the MLflow UI.

The scores illustrate the workflow, not model quality, and change slightly between runs.
The ten-experiment binary fit takes a few minutes; the others take about a minute each.

## Step 1. Where you start

The familiar Kumo Engine notebook trains a model with these lines:

```python
trainer, training_table, snapshot = example("binary", "many_trials")
result = trainer.fit(training_table, snapshot=snapshot)
result.summary()
```

`example(...)` builds a small seeded graph, training table and trainer; `trainer.fit(...)` trains.
Nothing is recorded anywhere except the model that `fit` returns.
Below, the same two calls run with MLOps on, and the `fit` call itself does not change.

## Step 2. Install

The documented install is:

```bash
pip install "kumo-engine[mlflow]"
```

The `mlflow` extra installs `mlflow-skinny` (the MLflow client) and `nvidia-ml-py` (GPU system metrics).
It does not install the MLflow server or UI.
The skinny package does not depend on Flask, SQLAlchemy or Alembic, so the machine that hosts the server needs the full `mlflow` package as well.
On this instance the checkout's environment already has Kumo Engine installed in editable mode with the extra, and full `mlflow`, so the cell below only checks them.
The check tests for `kumo.MLOps` rather than a version number, because this branch reports a development version.

In [1]:
import sys
from pathlib import Path

REPO = Path("/home/ubuntu/kumo-engine-fork")
assert Path(sys.prefix) == REPO / ".venv", sys.prefix  # the result-summary kernel
print("Kernel environment:", sys.prefix)

Kernel environment: /home/ubuntu/kumo-engine-fork/.venv


In [2]:
import importlib.metadata as md
import inspect
import warnings

# tqdm warns that ipywidgets is absent; fits here show no progress bars.
warnings.filterwarnings("ignore", message="IProgress not found")
import kumo.engine as kumo
import mlflow

print("kumo-engine", md.version("kumo-engine"), "imported from", Path(kumo.__file__).parent)
print("kumo.MLOps available:", hasattr(kumo, "MLOps"))
for package in ("mlflow", "mlflow-skinny", "nvidia-ml-py"):
    print(f"{package:14}", md.version(package))
assert hasattr(kumo, "MLOps")
assert Path(kumo.__file__).is_relative_to(REPO / "packages/kumo-engine")

kumo-engine 1.0.0.dev0 imported from /home/ubuntu/kumo-engine-fork/packages/kumo-engine/kumo/engine
kumo.MLOps available: True
mlflow         3.17.1
mlflow-skinny  3.17.1
nvidia-ml-py   13.615.71


## Step 3. The self-managed MLflow server

Kumo Engine writes two separate things.
The **artifact root** is a folder (or bucket) that holds Kumo's own snapshots, training tables, model bundles and predictions.
The **MLflow server** records runs, metrics, tags, logged models and the model registry, and serves the UI.

Here the server runs on this instance, on port 5000, in a tmux session.
Its backend store is a SQLite file and it keeps artifacts on local disk, proxied through the server so that clients only need its URL.
It was started with this command:

```bash
mlflow server \
  --backend-store-uri sqlite:////home/ubuntu/kumo-mlops/mlflow-server/mlflow.db \
  --artifacts-destination /home/ubuntu/kumo-mlops/mlflow-server/artifacts \
  --serve-artifacts \
  --host 127.0.0.1 --port 5000 \
  --allowed-hosts localhost:5000,127.0.0.1:5000,localhost,127.0.0.1
```

MLflow 3.17 and newer refuse a plain-folder backend store unless `MLFLOW_ALLOW_FILE_STORE=true` is set, and Kumo's own error message recommends SQLite, so this demo uses SQLite.

In [3]:
import requests
from mlflow import MlflowClient

MLFLOW_URL = "http://127.0.0.1:5000"
print("health:", requests.get(f"{MLFLOW_URL}/health", timeout=10).text)
print("MLflow client", mlflow.__version__)
print("experiments before training:",
      [(e.experiment_id, e.name) for e in MlflowClient(MLFLOW_URL).search_experiments()])
print("registered models before training:", MlflowClient(MLFLOW_URL).search_registered_models())

health: OK
MLflow client 3.17.1
experiments before training: [('0', 'Default')]


registered models before training: []


## Step 4. Turn MLOps on

One `kumo.MLOps` object says where Kumo's artifacts go, which tracker records runs, and which registry names the model.
`MlflowTracking(experiment, target=url)` points at the self-managed server; `on_error="raise"` makes any tracking or registry problem fail loudly instead of warning.
Building it already contacts the server: it creates the experiment if needed and checks that the artifact root is writable.

Then `with kumo.mlops(cfg):` turns it on for the calls inside the block: snapshot, training table, fit and predict.

Every task below logs to one experiment, `result-summary-demo`, and registers its own model (`demo-binary`, `demo-multiclass`, ...) under the alias `candidate`.
`MlflowTracking` is imported from `kumo.engine.integrations.mlflow`, not from `kumo.engine`.

In [4]:
from kumo.engine.integrations.mlflow import MlflowTracking

ARTIFACT_ROOT = "/home/ubuntu/kumo-mlops/kumo"
EXPERIMENT = "result-summary-demo"


def model_name(task):
    return "demo-" + task.replace("_", "-")


def mlops_for(task):
    return kumo.MLOps(
        ARTIFACT_ROOT,
        tracking=MlflowTracking(EXPERIMENT, target=MLFLOW_URL),
        registry=kumo.ModelRegistryLocation(model_name(task), alias="candidate"),
        on_error="raise",
    )


cfg = mlops_for("binary")
print(cfg.artifact_root)
print(cfg.tracking.experiment_name, "on", MLFLOW_URL)
print(cfg.registry)

/home/ubuntu/kumo-mlops/kumo
result-summary-demo on http://127.0.0.1:5000
ModelRegistryLocation(model_name='demo-binary', version=None, comment=None, alias='candidate')


The example data comes from the same generators as the `result.summary()` demo.
`example(task, scenario, cfg)` is the familiar `example(...)` with one addition: it builds the snapshot and training table inside `kumo.mlops(cfg)`, so Kumo stores them under the artifact root and records them as lineage.
Fits run inside `quiet()`, which hides their ingest and training logs; errors still raise.
Models are written under `~/result-summary-demo-runs`, outside the checkout.

In [5]:
import contextlib, importlib.util, logging, os, tempfile

warnings.filterwarnings("ignore")
spec = importlib.util.spec_from_file_location(
    "validation_harness", REPO / "artifacts/result-summary/validation_harness.py")
h = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h)
RUNS = Path.home() / "result-summary-demo-runs"
RUNS.mkdir(exist_ok=True)
OUT = Path(tempfile.mkdtemp(dir=RUNS))


@contextlib.contextmanager
def quiet():
    """Hide the ingest and training logs of a fit; errors still raise."""
    sys.stdout.flush(), sys.stderr.flush()
    saved = os.dup(1), os.dup(2)
    logging.disable(logging.CRITICAL)
    try:
        with open(os.devnull, "w") as null, contextlib.redirect_stdout(null), \
                contextlib.redirect_stderr(null):
            os.dup2(null.fileno(), 1), os.dup2(null.fileno(), 2)
            yield
    finally:
        logging.disable(logging.NOTSET)
        for fd, copy in zip((1, 2), saved):
            os.dup2(copy, fd), os.close(copy)


def example(task, scenario="baseline", cfg=None):
    """A trainer, training table and snapshot for one small task."""
    with kumo.mlops(cfg) if cfg else contextlib.nullcontext(), quiet():
        ctx = h.prepare(REPO, OUT / task, task, scenario, "mlops demo")
    return ctx["trainer"], ctx["table"], ctx["snapshot"]

## Step 5. Train

Binary classification first, with ten experiments (trials), so there is something to compare later.
The `trainer.fit(training_table, snapshot=snapshot)` line is exactly the one from Step 1.
Only the `with kumo.mlops(cfg):` around it is new.

In [6]:
cfg = mlops_for("binary")
trainer, training_table, snapshot = example("binary", "many_trials", cfg)  # stored under the artifact root
with kumo.mlops(cfg), quiet():
    result = trainer.fit(training_table, snapshot=snapshot)  # unchanged fit call

With MLOps on, the result carries two new fields.
`result.tracking` is the MLflow run of this fit, and `result.registered_model` is the registered model version.
`result.summary()` works unchanged.

In [7]:
display(result.tracking)
print("registered model:", result.registered_model.name, "version", result.registered_model.version,
      "->", result.registered_model.uri)
print("run id:", result.tracking.run_id)
print("run URL:", result.tracking.url)

TrackingRef(platform='mlflow', run_id='ff3ea92f9d784a448391cd58491d1709', url='http://127.0.0.1:5000/#/experiments/1/runs/ff3ea92f9d784a448391cd58491d1709', experiment='result-summary-demo', run_name='fit-demo-binary-v1')

registered model: demo-binary version 1 -> models:/demo-binary/1
run id: ff3ea92f9d784a448391cd58491d1709
run URL: http://127.0.0.1:5000/#/experiments/1/runs/ff3ea92f9d784a448391cd58491d1709


In [8]:
result.summary()

Evaluation metrics
Metric | Training | Validation | Holdout
AUROC * | 0.733 | 0.938 | 0.838
Accuracy | 0.507 | 0.745 | 0.780
AP | 0.734 | 0.948 | 0.765
AUPRC | 0.730 | 0.947 | 0.756
Brier score | - | - | 0.213
F1 | 0.027 | 0.711 | 0.667
Log loss | - | - | 0.623
Precision | 1.000 | 0.941 | 0.750
Recall | 0.014 | 0.571 | 0.600
Learning curves: Experiment 6 ★
  train loss: 0.708, 0.698, 0.690, 0.685
  val loss: 0.699, 0.692, 0.684, 0.674
  train auroc: 0.154, 0.269, 0.733, 0.836
  val auroc: 0.174, 0.865, 0.938, 0.930
No per-epoch history: Brier score, Log loss

Now the other seven task types, with the same two lines each.
Each fit registers its own model under the alias `candidate`.

In [9]:
ALL_TASKS = (
    ("binary", "many_trials"), ("multiclass", "baseline"), ("multilabel", "baseline"),
    ("multilabel_ranking", "baseline"), ("regression", "baseline"), ("forecasting", "baseline"),
    ("static_link", "baseline"), ("temporal_link", "baseline"),
)
FITS = {"binary": (cfg, trainer, snapshot, result)}
for task, scenario in ALL_TASKS[1:]:
    task_cfg = mlops_for(task)
    task_trainer, task_table, task_snapshot = example(task, scenario, task_cfg)
    with kumo.mlops(task_cfg), quiet():
        task_result = task_trainer.fit(task_table, snapshot=task_snapshot)
    FITS[task] = (task_cfg, task_trainer, task_snapshot, task_result)
    print(f"{task:20} {task_result.tracking.run_name:40} "
          f"{task_result.registered_model.name} v{task_result.registered_model.version}  "
          f"{task_result.tracking.url}")

multiclass           fit-demo-multiclass-v1                   demo-multiclass v1  http://127.0.0.1:5000/#/experiments/1/runs/b4779a98958d446ab63261c191260dfa


multilabel           fit-demo-multilabel-v1                   demo-multilabel v1  http://127.0.0.1:5000/#/experiments/1/runs/8257e0d9421f464ca8b23a15cad2100f


multilabel_ranking   fit-demo-multilabel-ranking-v1           demo-multilabel-ranking v1  http://127.0.0.1:5000/#/experiments/1/runs/cc7c079b2df94ac99478de4a485709c2


regression           fit-demo-regression-v1                   demo-regression v1  http://127.0.0.1:5000/#/experiments/1/runs/d88098b2996a466eadfc944525bedee8


forecasting          fit-demo-forecasting-v1                  demo-forecasting v1  http://127.0.0.1:5000/#/experiments/1/runs/097dcfc92a34489ba64d338c26acebd4


static_link          fit-demo-static-link-v1                  demo-static-link v1  http://127.0.0.1:5000/#/experiments/1/runs/7f91aa664d50462abd84655c81263514


temporal_link        fit-demo-temporal-link-v1                demo-temporal-link v1  http://127.0.0.1:5000/#/experiments/1/runs/55992eb051db46e0b840f67f0528e8d3


## Step 6. Find it in MLflow, from Python

Everything below reads from the MLflow server, not from the `result` objects.
`cfg.tracking.client()` gives an MLflow client for the server the config points at.

Each fit is one run tagged `kumo.kind = fit`; its trials are child runs tagged `kumo.kind = trial`.
This table compares the eight fits.

In [10]:
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)


def metric_key(name):
    """MLflow metric names write "@" as "_at_" and "acc" as "accuracy"."""
    return "accuracy" if name == "acc" else name.replace("@", "_at_")


with cfg.tracking.client() as client:
    experiment = client.get_experiment_by_name(EXPERIMENT)
    fits = client.search_runs([experiment.experiment_id], filter_string="tags.`kumo.kind` = 'fit'",
                              order_by=["attributes.start_time ASC"], max_results=100)
rows = []
for run in fits:
    tags, metrics = run.data.tags, run.data.metrics
    tune = tags["kumo.tune_metric"]
    extra = [k for k in sorted(metrics) if k.startswith("test/")
             and k not in (f"test/{metric_key(tune)}", "test/loss")][:3]
    rows.append({
        "run": run.info.run_name, "task": tags["kumo.task_type"], "tune metric": tune,
        "test (tune)": round(metrics.get(f"test/{metric_key(tune)}", float("nan")), 4),
        "other test metrics": ", ".join(f"{k[5:]}={metrics[k]:.3g}" for k in extra),
        "best trial": tags.get("kumo.best_trial"), "best epoch": tags.get("kumo.best_epoch"),
        "model": f"{tags.get('kumo.model_name')} v{tags.get('kumo.model_version')}",
    })
print("experiment", experiment.experiment_id, experiment.name, "-", len(fits), "fit runs")
pd.DataFrame(rows)

experiment 1 result-summary-demo - 8 fit runs


,run,task,tune metric,test (tune),other test metrics,best trial,best epoch,model
0,fit-demo-binary-v1,binary_classification,auroc,0.8385,"accuracy=0.78, ap=0.765, auprc=0.756",6,3,demo-binary v1
1,fit-demo-multiclass-v1,multiclass_classification,accuracy,0.5122,"f1=0.366, f1_macro=0.366, f1_micro=0.512",None,3,demo-multiclass v1
2,fit-demo-multilabel-v1,multilabel_classification,ap_macro,0.6558,"ap_micro=0.698, auprc_macro=0.635, auprc_micro=0.696",None,4,demo-multilabel v1
3,fit-demo-multilabel-ranking-v1,multilabel_ranking,recall@5,0.8972,"f1_at_1=0.359, f1_at_10=0.448, f1_at_5=0.632",None,1,demo-multilabel-ranking v1
4,fit-demo-regression-v1,regression,mae,1.3672,"bias=-0.224, mape=0.558, mse=2.83",None,4,demo-regression v1
5,fit-demo-forecasting-v1,forecasting,mae,1.3561,"bias=-0.107, mape=0.577, mse=2.82",None,4,demo-forecasting v1
6,fit-demo-static-link-v1,static_link_prediction,recall@5,0.8813,"f1_at_1=0.357, f1_at_10=0.629, f1_at_5=0.809",None,3,demo-static-link v1
7,fit-demo-temporal-link-v1,temporal_link_prediction,recall@10,0.3994,"f1_at_1=0.0576, f1_at_10=0.171, f1_at_5=0.14",None,4,demo-temporal-link v1


The ten trials of the binary fit are child runs of that fit.
They are compared by their best validation score, with the trial Kumo published marked ★.

In [11]:
binary_run_id = result.tracking.run_id
with cfg.tracking.client() as client:
    parent = client.get_run(binary_run_id)
    trials = client.search_runs([experiment.experiment_id],
                                filter_string=f"tags.`mlflow.parentRunId` = '{binary_run_id}'", max_results=100)
tune = parent.data.tags["kumo.tune_metric"]
best_trial = parent.data.tags["kumo.best_trial"]
rows = []
for trial in trials:
    number = trial.info.run_name.split("-")[1]
    rows.append({
        "trial": ("★ " if number == best_trial else "") + trial.info.run_name,
        "kind": trial.data.tags["kumo.kind"], "base_lr": float(trial.data.params["optimization.base_lr"]),
        f"best val/{tune}": round(trial.data.metrics[f"val/best_{tune}"], 4),
        "status": trial.data.tags.get("kumo.status"),
    })
frame = pd.DataFrame(rows).sort_values(f"best val/{tune}", ascending=False).reset_index(drop=True)
assert len(frame) == 10 and frame.trial.str.startswith("★").sum() == 1
frame

,trial,kind,base_lr,best val/auroc,status
0,trial-10-fit-demo-binary-v1,trial,0.1024,0.9379,succeeded
1,★ trial-6-fit-demo-binary-v1,trial,0.0008,0.9379,succeeded
2,trial-1-fit-demo-binary-v1,trial,0.0512,0.9363,succeeded
3,trial-4-fit-demo-binary-v1,trial,0.0512,0.9363,succeeded
4,trial-7-fit-demo-binary-v1,trial,0.0512,0.9348,succeeded
5,trial-8-fit-demo-binary-v1,trial,0.0008,0.9332,succeeded
6,trial-2-fit-demo-binary-v1,trial,0.0016,0.9270,succeeded
7,trial-5-fit-demo-binary-v1,trial,0.0008,0.9193,succeeded
8,trial-3-fit-demo-binary-v1,trial,0.0002,0.9146,succeeded
9,trial-9-fit-demo-binary-v1,trial,0.0004,0.8602,succeeded


One metric history.
Kumo logs every metric of every epoch with the epoch as the step, so MLflow can draw the curve.
The fit run holds the curves of the trial that was published.

In [12]:
with cfg.tracking.client() as client:
    history = client.get_metric_history(binary_run_id, f"val/{tune}")
    train_history = client.get_metric_history(binary_run_id, f"train/{tune}")
pd.DataFrame({"epoch": [m.step for m in history], f"train/{tune}": [round(m.value, 4) for m in train_history],
              f"val/{tune}": [round(m.value, 4) for m in history]})

,epoch,train/auroc,val/auroc
0,1,0.1538,0.1739
1,2,0.2686,0.8649
2,3,0.7334,0.9379
3,4,0.8358,0.9301


One fit run in detail: parameters, `kumo.*` tags, artifacts, dataset inputs (lineage) and the run description card.

In [13]:
with cfg.tracking.client() as client:
    run = client.get_run(binary_run_id)
    artifacts = client.list_artifacts(binary_run_id)
    data_files = client.list_artifacts(binary_run_id, "data")
    inputs = run.inputs.dataset_inputs
params = run.data.params
print(f"{len(params)} params, for example:")
for key in ("training_job.num_experiments", "training_job.tune_metric", "optimization.max_epochs", "optimization.base_lr"):
    print(f"  {key} = {params.get(key)}")
print("\nkumo.* tags:")
for key, value in sorted(run.data.tags.items()):
    if key.startswith("kumo."):
        print(f"  {key} = {value[:90]}")
print("\nprep.* tags (snapshot and training table built inside kumo.mlops):")
for key, value in sorted(run.data.tags.items()):
    if key.startswith("prep."):
        print(f"  {key} = {value}")
print("\nrun artifacts:", [a.path for a in artifacts], "| data/:", [a.path for a in data_files])
print("\ndataset inputs (lineage):")
for entry in inputs:
    print(f"  {entry.dataset.name:35} source={entry.dataset.source_type:6} context={[t.value for t in entry.tags]}")

36 params, for example:
  training_job.num_experiments = 10
  training_job.tune_metric = auroc
  optimization.max_epochs = 4
  optimization.base_lr = [0.0002, 0.0004, 0.0008, 0.0016, 0.0032, 0.0064, 0.0128, 0.0256, 0.0512, 0.1024]

kumo.* tags:
  kumo.best_epoch = 3
  kumo.best_trial = 6
  kumo.bundle_id = bfbe845f0139c13396c78543e7523143a46bc0df22b1e93074eec82d0f9c4745
  kumo.bundle_uri = /home/ubuntu/kumo-mlops/kumo/models/bfbe845f0139c13396c78543e7523143a46bc0df22b1e93074eec8
  kumo.engine_version = 1.0.0.dev0
  kumo.entity = ents
  kumo.kind = fit
  kumo.model_name = demo-binary
  kumo.model_version = 1
  kumo.snapshot_id = 75f05e6666210575286dc4839152a573ca6817d1f35015ae1259ce816b277a02@7392a5dbb4c8586d
  kumo.task_type = binary_classification
  kumo.tune_metric = auroc

prep.* tags (snapshot and training table built inside kumo.mlops):
  prep.snapshot.bytes = 2583175
  prep.snapshot.seconds = 0.6229383300014888
  prep.training_table.bytes = 21139
  prep.training_table.seconds = 0

In [14]:
from IPython.display import Markdown

Markdown(run.data.tags["mlflow.note.content"])

holdout auroc 0.838


Fit of `demo-binary` v1 for binary classification of ents; its curves and holdout metrics are from trial 6 of 10, epoch 3, the best by val auroc.

`PREDICT ents.FLAG = 1 FOR EACH ents.ID`

**Holdout metrics**

| metric | value |
| --- | ---: |
| test/auroc | 0.838 |
| test/accuracy | 0.78 |
| test/ap (average precision) | 0.765 |
| test/auprc (area under the precision-recall curve, trapezoidal) | 0.756 |
| test/brier | 0.213 |
| test/f1 | 0.667 |
| test/logloss (cross-entropy of the scores) | 0.623 |
| test/loss (the training objective) | 0.684 |
| test/precision | 0.75 |
| test/recall | 0.6 |

**Training data**

| split | rows | distinct ents (approx.) | positive rate |
| --- | ---: | ---: | ---: |
| train | 148 | 148 | 0.5 |
| val | 51 | 51 | 0.549 |
| test | 41 | 41 | 0.366 |

**Data preparation**

| step | time (s) | disk |
| --- | ---: | --- |
| Graph snapshot | 0.623 | 2.46 MiB |
| Training table | 0.0237 | 20.6 KiB |

**Lineage**

- `ents` v1795069677 (`/home/ubuntu/result-summary-demo-runs/tmp4qbx1yht/binary/data/ents.parquet`), `events` v1795069677 (`/home/ubuntu/result-summary-demo-runs/tmp4qbx1yht/binary/data/events.parquet`), `items` v1795069677 (`/home/ubuntu/result-summary-demo-runs/tmp4qbx1yht/binary/data/items.parquet`) → graph snapshot 2026-11-19-7392a5 → training table 2026-11-19-b6066c → model `bfbe845f` (`demo-binary` v1)

**Where things are**

| Artifact | Location |
| --- | --- |
| model bundle `bfbe845f` | `/home/ubuntu/kumo-mlops/kumo/models/bfbe845f0139c13396c78543e7523143a46bc0df22b1e93074eec82d0f9c4745` |
| training table `tt-b6066` | `/home/ubuntu/kumo-mlops/kumo/training_tables/tt-b6066c55c4ff7faa617f4f93cd5cd535` |
| snapshot `75f05e66` | `/home/ubuntu/kumo-mlops/kumo/snapshots/75f05e6666210575286dc4839152a573ca6817d1f35015ae1259ce816b277a02/7392a5dbb4c8586d` |
| source table `ents` | `/home/ubuntu/result-summary-demo-runs/tmp4qbx1yht/binary/data/ents.parquet` |
| source table `events` | `/home/ubuntu/result-summary-demo-runs/tmp4qbx1yht/binary/data/events.parquet` |
| source table `items` | `/home/ubuntu/result-summary-demo-runs/tmp4qbx1yht/binary/data/items.parquet` |

Full lineage: `_provenance.json`. Tables: `holdout/by_threshold.json`, `holdout/calibration.json`, `data/health.json`, `data/columns.json`.

*kumo rewrites everything above this line; editing it stops kumo's updates.*

---
<!-- /kumo:card sha256=623348c0ededa169 -->

The logged model and the registered versions.
Because the config has a registry, each fit logs an MLflow model (the Kumo bundle wrapped as a pyfunc model) and registers it as a version of a registered model.

In [15]:
with cfg.tracking.client() as client:
    logged = client.search_logged_models(experiment_ids=[experiment.experiment_id])
    registered = client.search_registered_models()
    versions = client.search_model_versions("name = 'demo-binary'")
print("logged models:", len(logged))
pd.DataFrame([{"model": m.name, "model id": m.model_id, "status": str(m.status).split(".")[-1],
               "source run": m.source_run_id[:8]} for m in logged])

logged models: 8


,model,model id,status,source run
0,demo-temporal-link,m-d6207bbde8984b57a45049d36cf87fa3,READY,55992eb0
1,demo-static-link,m-68c1a3e53154483bbf91ef37b024d2af,READY,7f91aa66
2,demo-forecasting,m-3c04852384494ae8b5a51ab8a67a3877,READY,097dcfc9
3,demo-regression,m-4e1bf27b275c483f93fe241012d44033,READY,d88098b2
4,demo-multilabel-ranking,m-f118d2e1c4b34d288a587ce476b127b7,READY,cc7c079b
5,demo-multilabel,m-fd0bd0aa6d0642539dc4da76fa9ddfc7,READY,8257e0d9
6,demo-multiclass,m-97de50ad351d4b798091ed347e7a4ebe,READY,b4779a98
7,demo-binary,m-a50b55978a9745f39ee33807b63f3256,READY,ff3ea92f


In [16]:
with cfg.tracking.client() as client:
    table = []
    for model in sorted(registered, key=lambda m: m.name):
        latest = client.search_model_versions(f"name = '{model.name}'")
        table.append({"registered model": model.name, "versions": [v.version for v in latest],
                      "aliases": dict(model.aliases), "run": client.get_run(latest[0].run_id).info.run_name})
version = versions[0]
print("demo-binary v" + version.version, "| source", version.source, "| bundle", version.tags["kumo_bundle_id"][:12])
print(version.description)
pd.DataFrame(table)

demo-binary v1 | source models:/m-a50b55978a9745f39ee33807b63f3256 | bundle bfbe845f0139
test/auroc 0.838 · test/logloss 0.623 · test/loss 0.684 · test/accuracy 0.78
Trained by `fit-demo-binary-v1`. Bundle `/home/ubuntu/kumo-mlops/kumo/models/bfbe845f0139c13396c78543e7523143a46bc0df22b1e93074eec82d0f9c4745`.
<!-- /kumo:version sha256=c243ac6bd8a1ea4b -->


,registered model,versions,aliases,run
0,demo-binary,[1],{'candidate': '1'},fit-demo-binary-v1
1,demo-forecasting,[1],{'candidate': '1'},fit-demo-forecasting-v1
2,demo-multiclass,[1],{'candidate': '1'},fit-demo-multiclass-v1
3,demo-multilabel,[1],{'candidate': '1'},fit-demo-multilabel-v1
4,demo-multilabel-ranking,[1],{'candidate': '1'},fit-demo-multilabel-ranking-v1
5,demo-regression,[1],{'candidate': '1'},fit-demo-regression-v1
6,demo-static-link,[1],{'candidate': '1'},fit-demo-static-link-v1
7,demo-temporal-link,[1],{'candidate': '1'},fit-demo-temporal-link-v1


## Step 7. Use it

Promote the binary model: moving the alias `champion` to version 1 is plain MLflow registry usage.
Then load it back through the registry with `kumo.Trainer.load_from_connector`, check it is the same bundle that was trained, and score with it.

In [17]:
with cfg.tracking.client() as client:
    client.set_registered_model_alias("demo-binary", "champion", result.registered_model.version)
    print("demo-binary aliases:", dict(client.get_registered_model("demo-binary").aliases))

with quiet():  # hides MLflow's artifact download progress bars
    champion = kumo.Trainer.load_from_connector(
        cfg, location=kumo.ModelRegistryLocation("demo-binary", alias="champion"))
print("loaded bundle:", champion.result.id[:12], "| trained bundle:", result.id[:12])
print("registered as:", champion.result.registered_model.name, "version", champion.result.registered_model.version)
assert champion.result.id == result.id

demo-binary aliases: {'candidate': '1', 'champion': '1'}


loaded bundle: bfbe845f0139 | trained bundle: bfbe845f0139
registered as: demo-binary version 1


The binary query is static, and without `entities=` Kumo scores only the entities whose label is still unknown.
Every entity in this demo data has a label, so we pass the entities to score: a Parquet file with an `ENTITY1` column, here the first fifty IDs.
The scoring call runs inside `kumo.mlops(cfg)`, which makes it a tracked prediction run.
In the result, `TARGET_PRED` is empty (`<NA>`) because no classification threshold was set; the class scores `False_PROB` and `True_PROB` are always returned.
To also get a predicted label, pass `binary_classification_threshold=` to `predict`, or set it when fitting with `scoring_defaults=kumo.ScoringDefaults(binary_classification_threshold=...)`.

In [18]:
to_score = OUT / "to-score"
to_score.mkdir()
pd.DataFrame({"ENTITY1": range(50)}).to_parquet(to_score / "entities.parquet", index=False)
entities = kumo.FileConnector(root_dir=str(to_score))["entities.parquet"]
with kumo.mlops(cfg), quiet():
    prediction = champion.predict(snapshot, entities=entities)
print("prediction", prediction.id[:12], "| rows:", prediction.num_rows)
print("tracked as:", prediction.tracking.run_name)
print("run URL:", prediction.tracking.url)
prediction.frame.head()

prediction 7aa8b52b0777 | rows: 50
tracked as: predict-demo-binary-v1-2026-11-19-062949
run URL: http://127.0.0.1:5000/#/experiments/1/runs/bb11c4be56214f7db6797ddf350f027b


,ENTITY,TARGET_PRED,False_PROB,True_PROB
0,0,<NA>,0.230671,0.769329
1,2,<NA>,0.061250,0.938750
2,4,<NA>,0.952760,0.047240
3,6,<NA>,0.143020,0.856980
4,8,<NA>,0.938286,0.061714


The prediction is its own run, tagged `kumo.kind = prediction`, linked back to the fit run and the model version.

In [19]:
with cfg.tracking.client() as client:
    predictions = client.search_runs([experiment.experiment_id],
                                     filter_string="tags.`kumo.kind` = 'prediction'")
assert len(predictions) == 1
pred_run = predictions[0]
print(pred_run.info.run_name)
for key, value in sorted(pred_run.data.tags.items()):
    if key.startswith("kumo."):
        print(f"  {key} = {value[:90]}")
print("  fit run:", binary_run_id, "-> same:", pred_run.data.tags["kumo.fit_run_id"] == binary_run_id)

predict-demo-binary-v1-2026-11-19-062949
  kumo.as_of = 2026-11-19
  kumo.bundle_id = bfbe845f0139c13396c78543e7523143a46bc0df22b1e93074eec82d0f9c4745
  kumo.bundle_uri = /home/ubuntu/kumo-mlops/kumo/models/bfbe845f0139c13396c78543e7523143a46bc0df22b1e93074eec8
  kumo.engine_version = 1.0.0.dev0
  kumo.entity = ents
  kumo.fit_run_id = ff3ea92f9d784a448391cd58491d1709
  kumo.kind = prediction
  kumo.model_name = demo-binary
  kumo.model_version = 1
  kumo.output_uri = /home/ubuntu/kumo-mlops/kumo/predictions/7aa8b52b07774212930ab9c1fe6d1b48-predict-05df327f
  kumo.prediction_id = 7aa8b52b07774212930ab9c1fe6d1b48-predict-05df327f3909472684bf2ad822bcc0ae
  kumo.snapshot_id = 75f05e6666210575286dc4839152a573ca6817d1f35015ae1259ce816b277a02@7392a5dbb4c8586d
  kumo.task_type = binary_classification
  kumo.tune_metric = auroc
  fit run: ff3ea92f9d784a448391cd58491d1709 -> same: True


Kumo's own files live under the artifact root.
These are its top two levels.

In [20]:
def size(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())


root = Path(ARTIFACT_ROOT)
for top in sorted(p for p in root.iterdir() if p.is_dir()):
    children = sorted(top.iterdir())
    print(f"{top.name + '/':18} {len(children):2} entries, {size(top) / 1e6:7.1f} MB")
    for child in children[:3]:
        print(f"    {child.name[:60]}")
    if len(children) > 3:
        print(f"    ... {len(children) - 3} more")

models/             8 entries,     1.8 MB
    0a0f66867717b9977068158ed6a05495a8de6d880f8a26a67e165adcf740
    15861521de036aca19df276ac4d3bc6f8133a07217ca5e9ca8450209a8e4
    31384175d5604d26d593715c69fcae74269479a1d25872b27cf6bb74c2ad
    ... 5 more
predictions/        1 entries,     0.0 MB
    7aa8b52b07774212930ab9c1fe6d1b48-predict-05df327f3909472684b
snapshots/          8 entries,    20.4 MB
    1dffab8eb811ee2db9fb024baa907867d537bd1aebb53d5ea97c808af695
    6d32cb8015e1d16a02dfe702b05d7da4c6ca20ed697ea7b379af1121d285
    75f05e6666210575286dc4839152a573ca6817d1f35015ae1259ce816b27
    ... 5 more
training_tables/    8 entries,     0.1 MB
    tt-45ead35884bf4f9feef959e7b7fd00f8
    tt-b6066c55c4ff7faa617f4f93cd5cd535
    tt-caefbe75aa41802c6a09768f3fcbcc24
    ... 5 more


## Step 8. Open the UI

The server listens on `http://127.0.0.1:5000` on the instance.
From a laptop, forward the port first, then open `http://localhost:5000`:

```bash
ssh -N -L 5000:127.0.0.1:5000 kumo-engine-experimental
```

MLflow 3 opens its GenAI views first.
Adding `?workflowType=machine_learning` to a link opens it in the classic **Model training** view, which the links below do.

In [21]:
import json
from IPython.display import HTML, display

experiment_id = experiment.experiment_id
binary_model = next(m for m in logged if m.name == "demo-binary")
ML = "workflowType=machine_learning"  # opens MLflow's classic Model training views
trial_ids = [trial.info.run_id for trial in trials]
links = {
    "Experiment, Model training view": f"{MLFLOW_URL}/#/experiments/{experiment_id}/runs?{ML}",
    "Binary fit run": f"{MLFLOW_URL}/#/experiments/{experiment_id}/runs/{binary_run_id}?{ML}",
    "Binary trials compared": f"{MLFLOW_URL}/#/compare-runs?runs={json.dumps(trial_ids)}&experiments={json.dumps([experiment_id])}&{ML}",
    "Prediction run": f"{MLFLOW_URL}/#/experiments/{experiment_id}/runs/{pred_run.info.run_id}?{ML}",
    "Logged model": f"{MLFLOW_URL}/#/experiments/{experiment_id}/models/{binary_model.model_id}?{ML}",
    "Registered model demo-binary": f"{MLFLOW_URL}/#/models/demo-binary?{ML}",
}
for label, url in links.items():
    print(f"{label:32} {url}")
display(HTML("<br>".join(f'<a href="{url}" target="_blank" rel="noopener">{label}</a>' for label, url in links.items())))

Experiment, Model training view  http://127.0.0.1:5000/#/experiments/1/runs?workflowType=machine_learning
Binary fit run                   http://127.0.0.1:5000/#/experiments/1/runs/ff3ea92f9d784a448391cd58491d1709?workflowType=machine_learning
Binary trials compared           http://127.0.0.1:5000/#/compare-runs?runs=["b66ca6d5bddd4209ba8846e29f98afc1", "5ed32aaaa1ab4220918d16c78477ffe9", "97bb6cd9df844cc5bb20500ec51991ca", "2d269dda4bee43f1b9550819f8a3a9ab", "d08c4c81e30943d1b6d9771bab523819", "6e070e77ac544f879489fbbac56f354a", "73e8542c51ee4f12a013e12523a018f3", "1ebb31f06b4749efab88b8d1d378c8f5", "a8257d4fdd4d469f890ab41169908fdd", "faf5e037254f4435b7b90cee01385bef"]&experiments=["1"]&workflowType=machine_learning
Prediction run                   http://127.0.0.1:5000/#/experiments/1/runs/bb11c4be56214f7db6797ddf350f027b?workflowType=machine_learning
Logged model                     http://127.0.0.1:5000/#/experiments/1/models/m-a50b55978a9745f39ee33807b63f3256?workflowType=machin